# Given a sim, investigate flux through the metabolic network by plotting a sankey diagram of fluxes through reactions. This is a more detailed version of the reaction usage plot, which only plots the number of reactions used and kinetic target scatter.

In [1]:
import pandas as pd
import os, dill
import numpy as np
import cvxpy as cp
from typing import Iterable, Optional, Mapping, cast, Any
from plotly import graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from yaml import emit

os.chdir(os.path.expanduser('~/dev/vEcoli'))

%load_ext autoreload

In [2]:
def load_sim(
        time_num: int,
        date: str,
        experiment_name: str,
        condition: str,
        experiment_type: str,
):
    # --- Load Sim ---
    time = str(time_num)
    entry = f'{experiment_name}_{time}_{date}'
    folder_path = f'out/{experiment_type}/{condition}/{entry}/'

    output = np.load(folder_path + '0_output.npy',allow_pickle='TRUE').item()
    output = output['agents']['0']
    fba = output['listeners']['fba_results']
    bulk = pd.DataFrame(output['bulk'])
    f = open(folder_path + 'agent_steps.pkl', 'rb')
    agent = dill.load(f)
    f.close()

    metabolism = agent['ecoli-metabolism-redux-classic']

    return fba, bulk, metabolism, output

In [3]:
def hex_to_rgba(hex_color, alpha=0.4):
    hex_color = hex_color.lstrip("#")
    r, g, b = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

In [4]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

def sankey_metabolite_flux(
    met_of_interest,
    fba: dict,
    metabolism: object,
    *,
    timestep=None,
    agg: str = "mean",
    top_n: int = 12,
    min_abs_contrib: float | None = None,
    title: str | None = None,
):
    """
    Sankey for ONE metabolite using user's get_subset_S().

    Returns
    -------
    fig : plotly.graph_objects.Figure
    contrib_table : pd.DataFrame
    """
    S = metabolism.stoichiometry.copy()
    reaction_names = metabolism.reaction_names
    metabolites = metabolism.metabolite_names
    S = pd.DataFrame(S, index=metabolites, columns=reaction_names)
    flux = pd.DataFrame(fba['estimated_fluxes'], columns=reaction_names)
    # --- user's helper ---
    def get_subset_S(S, met_of_interest):
        S_met = S.loc[met_of_interest, :]
        S_met = S_met.loc[:, ~np.all(S_met == 0, axis=0)]
        return S_met, S_met.columns

    # Normalize met input
    if isinstance(met_of_interest, str):
        met_list = [met_of_interest]
    else:
        met_list = list(met_of_interest)

    if len(met_list) != 1:
        raise ValueError("Please pass exactly one metabolite name (string or single-item list).")
    met = met_list[0]

    # Get stoich row subset (1 x rxns) and participating rxns
    S_met, rxns = get_subset_S(S, [met])  # returns DataFrame with 1 row
    s = S_met.loc[met]                    # Series indexed by rxns

    # --- pick a single flux vector v (Series indexed by rxns) ---
    if isinstance(flux, pd.DataFrame):
        # restrict to participating rxns first to avoid reindex noise
        flux_sub = flux.loc[:, rxns]

        if timestep is not None:
            v = flux_sub.loc[timestep]
        else:
            if agg == "mean":
                v = flux_sub.mean(axis=0)
            elif agg == "sum":
                v = flux_sub.sum(axis=0)
            elif agg == "median":
                v = flux_sub.median(axis=0)
            elif agg == "max":
                v = flux_sub.max(axis=0)
            elif agg == "min":
                v = flux_sub.min(axis=0)
            else:
                raise ValueError(f"Unknown agg='{agg}'")
    elif isinstance(flux, pd.Series):
        v = flux.reindex(rxns).fillna(0.0)
    else:
        raise TypeError("flux must be a pandas Series or DataFrame")

    # Signed contribution to d(met)/dt
    contrib = s * v
    abs_contrib = contrib.abs()

    contrib_table = pd.DataFrame(
        {
            "stoich": s,
            "flux": v,
            "contrib": contrib,
            "abs_contrib": abs_contrib,
        }
    ).sort_values("abs_contrib", ascending=False)

    if min_abs_contrib is not None:
        contrib_table = contrib_table[contrib_table["abs_contrib"] >= float(min_abs_contrib)]

    # Producers/consumers by SIGN of contribution
    producers = contrib_table[contrib_table["contrib"] > 0].head(top_n)
    consumers = contrib_table[contrib_table["contrib"] < 0].head(top_n)

    if producers.empty and consumers.empty:
        raise ValueError(
            f"No nonzero contributions found for '{met}' "
            f"(after filtering). Try a different timestep/agg or lower min_abs_contrib."
        )

    # --- Sankey nodes ---
    prod_names = producers.index.tolist()
    cons_names = consumers.index.tolist()
    nodes = prod_names + [met] + cons_names
    idx = {name: i for i, name in enumerate(nodes)}
    met_idx = idx[met]

    # --- Sankey links ---
    sources, targets, values, link_labels = [], [], [], []

    for r, row in producers.iterrows():
        sources.append(idx[r])
        targets.append(met_idx)
        values.append(float(row["abs_contrib"]))
        link_labels.append(f"{r} produces {met}: |S*v|={row['abs_contrib']:.3g}")

    for r, row in consumers.iterrows():
        sources.append(met_idx)
        targets.append(idx[r])
        values.append(float(row["abs_contrib"]))
        link_labels.append(f"{r} consumes {met}: |S*v|={row['abs_contrib']:.3g}")

    # --- Sankey color ---
    colors = px.colors.qualitative.Pastel

    # Assign a color to each node, cycling through the palette
    node_colors = [colors[i % len(colors)] for i in range(len(nodes))]

    # Optionally, give links the color of their source node
    link_colors = [node_colors[s] for s in sources]

    # --- Plot ---
    fig = go.Figure(
        data=[
            go.Sankey(
                arrangement="snap",
                node=dict(
                    pad=12,
                    thickness=16,
                    line=dict(width=0.5),
                    label=nodes,
                    color=node_colors,
                ),
                link=dict(
                    source=sources,
                    target=targets,
                    value=values,
                    label=link_labels,
                    color=link_colors,
                ),
            )
        ]
    )

    if title is None:
        if isinstance(flux, pd.DataFrame) and timestep is not None:
            title = f"{met}: production/consumption contributions (t={timestep})"
        elif isinstance(flux, pd.DataFrame):
            title = f"{met}: production/consumption contributions ({agg} over time)"
        else:
            title = f"{met}: production/consumption contributions"

    fig.update_layout(title=title, template="plotly_white",
                      paper_bgcolor='rgba(255, 0, 0, 0)',
                      plot_bgcolor='rgba(255, 0, 0, 0)',)
    return fig, contrib_table


# Load Sim and Start Plotting

In [17]:
# import sim
time_num = 600
date = '2026-02-24'
experiment_name = 'homeo_diversity_8.53E-3'
condition = 'basal'
experiment_type = 'objective_weight'

fba, bulk, metabolism, output = load_sim(time_num, date, experiment_name, condition, experiment_type)

In [22]:
met_of_interest = 'DCTP[c]'
fig = sankey_metabolite_flux(
    met_of_interest=met_of_interest,
    fba=fba,
    metabolism=metabolism,
    agg="mean",
    top_n=10,
    min_abs_contrib=1e-6,
    title=f"{met_of_interest} production/consumption contributions (mean over time)",
)[0]
fig.show()

folder = f'out/{experiment_type}/{condition}/{experiment_name}_{time_num}_{date}/'
save_path = f'{folder}figures/'
if not os.path.exists(save_path):
    os.makedirs(save_path)
    print(f"Directory '{save_path}' created")

# fig.write_image(f'{save_path}sankey_CYTIDINE.png', width=1000, height=500, scale=3)

In [19]:
# import sim
time_num = 600
date = '2026-02-27'
experiment_name = 'homeo_diversity_8.53E-3'
condition = 'basal'
experiment_type = 'objective_weight'

fba_pr, bulk_pr, metabolism_pr, output_pr = load_sim(time_num, date, experiment_name, condition, experiment_type)

In [24]:
met_of_interest = 'DCTP[c]'
fig = sankey_metabolite_flux(
    met_of_interest=met_of_interest,
    fba=fba_pr,
    metabolism=metabolism_pr,
    agg="mean",
    top_n=10,
    min_abs_contrib=1e-6,
    title=f"{met_of_interest} production/consumption contributions (mean over time)",
)[0]
fig.show()

In [12]:
def get_subset_S(S, met_of_interest):
    S_met = S.loc[met_of_interest, :]
    S_met = S_met.loc[:,~np.all(S_met == 0, axis=0)]
    return S_met, S_met.columns

In [13]:
sim_flux = pd.DataFrame(fba['estimated_fluxes'], columns=metabolism.reaction_names)


# plot a plotly histogram of flux for each reaction of interest
met_of_interest = ['CYTIDINE[c]']
S = pd.DataFrame(metabolism.stoichiometry, index=metabolism.metabolite_names, columns=metabolism.reaction_names)
S_met, reactions = get_subset_S(S, met_of_interest)
flux_of_interest = sim_flux[reactions].mean(axis=0)

fig = px.bar(
    x=flux_of_interest.index,
    y=flux_of_interest.values,
    title=f"Mean Fluxes for Reactions Involving {met_of_interest[0]} (Before PR)",
    labels={"x": "Reaction", "y": "Mean Flux (mmol/L/s)"},
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig.update_layout(template="plotly_white",
                  showlegend=False,
                  paper_bgcolor='rgba(255, 0, 0, 0)',
                  plot_bgcolor='rgba(255, 0, 0, 0)',
                  title_font_size=20,)
fig.show(renderer='browser')

folder = f'out/{experiment_type}/{condition}/{experiment_name}_{time_num}_{date}/'
save_path = f'{folder}figures/'
if not os.path.exists(save_path):
    os.makedirs(save_path)
    print(f"Directory '{save_path}' created")

# fig.write_image(f'{save_path}flux_to_CYTIDINE.png', width=900, height=600, scale=3)


In [97]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# --- Build Sankey trace ---
met_of_interest = 'CYTIDINE[c]'
sankey_fig, _ = sankey_metabolite_flux(
    met_of_interest=met_of_interest,
    fba=fba,
    metabolism=metabolism,
    agg="mean",
    top_n=10,
    min_abs_contrib=1e-6,
    title=f"{met_of_interest} production/consumption contributions (mean over time)",
)
sankey_trace = sankey_fig.data[0]  # the go.Sankey object

# --- Build Bar trace ---
sim_flux = pd.DataFrame(fba['estimated_fluxes'], columns=metabolism.reaction_names)

S = pd.DataFrame(metabolism.stoichiometry, index=metabolism.metabolite_names, columns=metabolism.reaction_names)
S_met, reactions = get_subset_S(S, [met_of_interest])
flux_of_interest = sim_flux[reactions].mean(axis=0)

bar_trace = go.Bar(
    x=flux_of_interest.index,
    y=flux_of_interest.values,
    marker_color=colors[0],
    name="Mean Flux",
)

# --- Combine into 2x1 subplot ---
fig = make_subplots(
    rows=2, cols=1,
    specs=[
        [{"type": "sankey"}],
        [{"type": "xy"}],
    ],
    subplot_titles=[
        f"{met_of_interest} production/consumption contributions (mean over time)",
        f"Mean Fluxes for Reactions Involving {met_of_interest}",
    ],
    row_heights=[1/3, 2/3],
    vertical_spacing=0.1,
)


fig.add_trace(sankey_trace, row=1, col=1)
fig.add_trace(bar_trace, row=2, col=1)
fig.update_layout(
    template="plotly_white",
    showlegend=False,
    height=1000,
    title_font_size=20,
    yaxis_title="Mean Flux (mmol/L/s)",
    xaxis_title="Reaction",
    # xaxis=dict(domain=[0.0, 0.75], ),  # bar chart takes 75% of  an
    paper_bgcolor='rgba(255, 0, 0, 0)',
    plot_bgcolor='rgba(255, 0, 0, 0)',
)

# fig.show()

# --- Save ---
folder = f'out/{experiment_type}/{condition}/{experiment_name}_{time_num}_{date}/'
save_path = f'{folder}figures/'
if not os.path.exists(save_path):
    os.makedirs(save_path)
    print(f"Directory '{save_path}' created")

# fig.write_image(f'{save_path}sankey_and_flux_CYTIDINE.png', width=1000, height=1000, scale=3)
fig.write_html(f'{save_path}sankey_and_flux_CYTIDINE.html')

In [96]:
fig.frames

()

In [11]:
sim_flux_pr = pd.DataFrame(fba_pr['estimated_fluxes'], columns=metabolism_pr.reaction_names)

# plot a plotly histogram of flux for each reaction of interest
met_of_interest = ['CYTIDINE[c]']
S = pd.DataFrame(metabolism_pr.stoichiometry, index=metabolism_pr.metabolite_names, columns=metabolism_pr.reaction_names)
S_met, reactions = get_subset_S(S, met_of_interest)
flux_of_interest = sim_flux_pr[reactions].mean(axis=0)

fig = px.bar(
    x=flux_of_interest.index,
    y=flux_of_interest.values,
    title=f"Mean Fluxes for Reactions Involving {met_of_interest[0]} (Before PR)",
    labels={"x": "Reaction", "y": "Mean Flux (mmol/L/s)"},
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig.update_layout(template="plotly_white",
                  showlegend=False,
                  paper_bgcolor='rgba(255, 0, 0, 0)',
                  plot_bgcolor='rgba(255, 0, 0, 0)',
                  title_font_size=20,)
# fig.show(renderer='browser')

folder = f'out/{experiment_type}/{condition}/{experiment_name}_{time_num}_{date}/'
save_path = f'{folder}figures/'
if not os.path.exists(save_path):
    os.makedirs(save_path)
    print(f"Directory '{save_path}' created")

# fig.write_image(f'{save_path}flux_to_CYTIDINE.png', width=1000, height=800, scale=3)


NameError: name 'get_subset_S' is not defined

In [17]:
folder

'out/objective_weight/basal/homeostatic_only_600_2026-01-22/'

In [21]:
metabolism.homeostatic_metabolites

array(['2-3-DIHYDROXYBENZOATE[c]', '2-KETOGLUTARATE[c]', '2-PG[c]',
       '2K-4CH3-PENTANOATE[c]', '4-AMINO-BUTYRATE[c]',
       '4-hydroxybenzoate[c]', 'ACETOACETYL-COA[c]', 'ACETYL-COA[c]',
       'ACETYL-P[c]', 'ADENINE[c]', 'ADENOSINE[c]', 'ADP-D-GLUCOSE[c]',
       'ADP[c]', 'AMP[c]', 'ANTHRANILATE[c]', 'APS[c]', 'ARG[c]',
       'ASN[c]', 'ATP[c]', 'BIOTIN[c]', 'CA+2[c]', 'CAMP[c]',
       'CARBAMYUL-L-ASPARTATE[c]', 'CARBON-DIOXIDE[c]', 'CDP[c]',
       'CHORISMATE[c]', 'CIS-ACONITATE[c]', 'CIT[c]', 'CL-[c]', 'CMP[c]',
       'CO+2[c]', 'CO-A[c]', 'CPD-12115[c]', 'CPD-12261[p]',
       'CPD-12575[c]', 'CPD-12819[c]', 'CPD-12824[c]', 'CPD-13469[c]',
       'CPD-2961[c]', 'CPD-8260[c]', 'CPD-9956[c]', 'CPD0-939[c]',
       'CTP[c]', 'CYS[c]', 'CYTIDINE[c]', 'CYTOSINE[c]', 'D-ALA-D-ALA[c]',
       'D-SEDOHEPTULOSE-7-P[c]', 'DAMP[c]', 'DATP[c]', 'DCTP[c]',
       'DEOXY-RIBOSE-5P[c]', 'DEOXYADENOSINE[c]', 'DEOXYGUANOSINE[c]',
       'DGMP[c]', 'DGTP[c]', 'DI-H-OROTATE[c]',
       '